# Tutorial 7: Sensitivity

`Sensitivity` wraps any observable type (Zernikes, Spots, DoubleZernikes, …)
and provides the **linear relationship** between DOF perturbations and that
observable.  It stores:

- **nominal** — the unperturbed observable
- **gradient** — per-DOF derivatives (same type, with a leading DOF dimension)

This enables linear prediction: `nominal + gradient @ state` via `.predict(state)`.

## Key concepts

- **Generic typing**: `Sensitivity[Zernikes]`, `Sensitivity[Spots]`, etc. (all the same class)
- **Basis conversions**: `.x`, `.f`, `.v` — converts the gradient's DOF axis
- **Frame conversions**: `.ocs`, `.ccs`, etc. — propagates through gradient & nominal
- **Construction**: `Sensitivity.from_finite_differences()` or direct

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle
import galsim

from StarSharp import (
    FieldCoords, Zernikes, Spots, Sensitivity, State, StateFactory,
)

%matplotlib inline

## 1. Building a Sensitivity from finite differences

The standard construction is `Sensitivity.from_finite_differences()`.  You
provide a nominal observable, a list of perturbed observables (one per DOF),
and the step sizes used.

$$\text{gradient}_i = \frac{\text{perturbed}_i - \text{nominal}}{\text{step}_i}$$

In [ ]:
rtp = Angle(30, unit=u.deg)
rng = np.random.default_rng(42)

# Setup: 5 field points, jmax=15, 10 DOFs
n_field = 5
jmax = 15
n_dof = 10
R_outer = 4.18 * u.m
R_inner = 2.56 * u.m

field = FieldCoords(
    x=rng.uniform(-1, 1, n_field) * u.deg,
    y=rng.uniform(-1, 1, n_field) * u.deg,
    frame="ocs", rtp=rtp,
)

# Nominal Zernikes (unperturbed)
nominal_coefs = rng.normal(0, 0.05, (n_field, jmax + 1)) * u.um
nominal_coefs[:, :4] = 0 * u.um
nominal = Zernikes(
    coefs=nominal_coefs, field=field,
    R_outer=R_outer, R_inner=R_inner, frame="ocs", rtp=rtp,
)

# Simulate perturbed observations: each DOF shifts the Zernikes slightly
step_size = 1.0  # arbitrary units for step
steps = State(
    value=np.full(n_dof, step_size),
    basis="x",
    use_dof=np.arange(n_dof),
    n_dof=n_dof,
)

perturbed_list = []
for i in range(n_dof):
    # Each DOF adds a specific pattern to the Zernikes
    delta = np.zeros((n_field, jmax + 1))
    delta[:, 4 + i] = 0.01 * (i + 1)  # DOF-specific perturbation
    pert_coefs = nominal_coefs + delta * step_size * u.um
    perturbed_list.append(
        Zernikes(coefs=pert_coefs, field=field,
                 R_outer=R_outer, R_inner=R_inner, frame="ocs", rtp=rtp)
    )

sens = Sensitivity.from_finite_differences(nominal, perturbed_list, steps)
print(f"Type: {type(sens).__name__}")
print(f"Gradient type: {type(sens.gradient).__name__}")
print(f"Gradient coefs shape: {sens.gradient.coefs.shape}  →  (n_dof, n_field, jmax+1)")
print(f"Basis: {sens.basis}")
print(sens)

## 2. Linear prediction with `.predict(state)`

Given a DOF perturbation as a `State`, predict the resulting observable:

$$\text{predicted} = \text{nominal} + \sum_i \text{gradient}_i \cdot \text{state}_i$$

In [ ]:
# Create a state perturbation
perturbation = State(
    value=rng.normal(0, 0.5, n_dof),
    basis="x",
    use_dof=np.arange(n_dof),
    n_dof=n_dof,
)

# Predict the perturbed Zernikes
predicted = sens.predict(perturbation)
print(f"Predicted type: {type(predicted).__name__}")
print(f"Predicted coefs shape: {predicted.coefs.shape}")
print(f"Predicted == nominal? {np.allclose(predicted.coefs, nominal.coefs)}")
print(f"Max deviation from nominal: {np.max(np.abs(predicted.coefs - nominal.coefs)):.4f}")

In [ ]:
# Zero state → nominal
zero_state = State(
    value=np.zeros(n_dof), basis="x",
    use_dof=np.arange(n_dof), n_dof=n_dof,
)
pred_zero = sens.predict(zero_state)
print(f"predict(zero) == nominal: {np.allclose(pred_zero.coefs, nominal.coefs)}")

## 3. Frame conversions

Frame conversions propagate through both the gradient and the nominal.

In [ ]:
sens_ccs = sens.ccs
print(f"Original frame: {sens.frame}")
print(f"CCS frame: {sens_ccs.frame}")
print(f"Nominal frame: {sens_ccs.nominal.frame}")
print(f"Gradient frame: {sens_ccs.gradient.frame}")

In [ ]:
# Prediction is frame-consistent:
# Predicting in OCS and converting, vs. converting sensitivity then predicting
pred_ocs = sens.predict(perturbation)
pred_from_ccs = sens.ccs.predict(perturbation)

# These should give the same result (one in OCS, one in CCS)
print(f"OCS prediction frame: {pred_ocs.frame}")
print(f"CCS prediction frame: {pred_from_ccs.frame}")
print(f"Match after rotation: {np.allclose(pred_ocs.ccs.coefs, pred_from_ccs.coefs)}")

## 4. Basis conversions

The gradient's leading (DOF) axis can be converted between bases.
This re-expresses the same linear relationship in a different DOF
parameterization.

In [ ]:
print(f"Original basis: {sens.basis}")
print(f"Gradient shape (x): {sens.gradient.coefs.shape}")

In [ ]:
# x → f : full DOF representation (pads with zeros for inactive DOFs)
sens_f = sens.f
print(f"f basis: gradient shape = {sens_f.gradient.coefs.shape}")

In [ ]:
# x → v : SVD-truncated basis (requires Vh)
# First attach Vh to the sensitivity
A_design = sens.gradient.coefs.to_value(u.um).reshape(n_dof, -1).T
U, S, Vh = np.linalg.svd(A_design, full_matrices=False)
nkeep = 5

sens_with_vh = Sensitivity(
    gradient=sens.gradient,
    nominal=sens.nominal,
    basis="x",
    use_dof=np.arange(n_dof),
    n_dof=n_dof,
    Vh=Vh[:nkeep],
)

sens_v = sens_with_vh.v
print(f"v basis: gradient shape = {sens_v.gradient.coefs.shape}")
print(f"nkeep = {nkeep}, reduced from {n_dof} DOFs")

## 5. Sensitivity with Spots

`Sensitivity` is generic — it works with any observable that defines
`_sensitivity_fields`.  Here we show it with `Spots`.

In [ ]:
n_ray = 100
n_dof_spots = 5

# Nominal spots
nom_spots = Spots(
    dx=rng.normal(0, 3, (n_field, n_ray)) * u.um,
    dy=rng.normal(0, 3, (n_field, n_ray)) * u.um,
    vignetted=np.zeros((n_field, n_ray), dtype=bool),
    field=field,
    frame="ccs", rtp=rtp,
)

# Perturbed spots (each DOF shifts the pattern)
pert_spots = []
for i in range(n_dof_spots):
    delta_dx = rng.normal(0, 0.5, (n_field, n_ray)) * u.um
    delta_dy = rng.normal(0, 0.5, (n_field, n_ray)) * u.um
    pert_spots.append(Spots(
        dx=nom_spots.dx + delta_dx,
        dy=nom_spots.dy + delta_dy,
        vignetted=nom_spots.vignetted,
        field=field, frame="ccs", rtp=rtp,
    ))

spot_steps = State(
    value=np.ones(n_dof_spots), basis="x",
    use_dof=np.arange(n_dof_spots), n_dof=n_dof_spots,
)

spot_sens = Sensitivity.from_finite_differences(nom_spots, pert_spots, spot_steps)
print(f"Sensitivity type: {spot_sens}")
print(f"Gradient dx shape: {spot_sens.gradient.dx.shape}  →  (n_dof, n_field, n_ray)")

In [ ]:
# Predict spots for a given state
spot_state = State(
    value=rng.normal(0, 1, n_dof_spots), basis="x",
    use_dof=np.arange(n_dof_spots), n_dof=n_dof_spots,
)
pred_spots = spot_sens.predict(spot_state)

print(f"Predicted spots type: {type(pred_spots).__name__}")
print(f"Predicted dx shape: {pred_spots.dx.shape}")

## 6. Bridge: Sensitivity → StateFactory

`StateFactory` can accept a `Sensitivity` directly — it extracts the design
matrix from the gradient's sensitivity fields, flattens, and concatenates them.

In [ ]:
sf = StateFactory(
    A=sens,  # pass the Sensitivity directly
    use_dof=np.arange(n_dof),
    nkeep=5,
)

print(f"Design matrix extracted from Sensitivity")
print(f"A shape: {sf.A.shape}  →  (n_obs, n_dof)")
print(f"Singular values: {sf.S}")
print(f"Vh shape: {sf.Vh.shape}")

## 7. Visualizing the gradient

In [ ]:
# Gradient matrix visualization: sensitivity of each Zernike to each DOF
# Average over field points for a compact view
grad_matrix = sens.gradient.coefs.to_value(u.um).mean(axis=1)  # (n_dof, jmax+1)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(
    grad_matrix[:, 4:], aspect="auto",
    cmap="viridis", origin="lower",
)
ax.set_xlabel("Noll j (starting from 4)")
ax.set_ylabel("DOF index")
ax.set_title("Mean sensitivity: ∂Zernike_j / ∂DOF_i")
plt.colorbar(im, ax=ax, label="[µm / step]")

## 8. Summary

| Property / Method | Description |
|---|---|
| `.nominal` | Unperturbed observable |
| `.gradient` | Per-DOF derivatives (leading dim = DOF) |
| `.predict(state)` | Linear prediction: `nominal + gradient @ state` |
| `.from_finite_differences(...)` | Construct from perturbed observables |
| `.basis` | DOF basis: `'x'`, `'f'`, or `'v'` |
| `.x`, `.f`, `.v` | Basis conversion (on DOF axis) |
| `.ocs`, `.ccs`, ... | Frame conversion (on observable) |
| `StateFactory(A=sens)` | **Bridge to StateFactory** (extracts design matrix) |

**Next:** [08_RaytracedOpticalModel.ipynb](08_RaytracedOpticalModel.ipynb) —
putting it all together with a real optical model.